# evaluate the authors’ pretrained cross-dataset models before training

In [1]:
from pathlib import Path
import yaml

# ============================================================
# PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/media/data/rPPG/Code/GitHub/Catch_The_Mamba"
)

OFFICIAL_ROOT = PROJECT_ROOT / "official" / "RhythmMamba"

MAMBA_HUNT_ROOT = Path(
    "/media/data/rPPG/rPPG_Data/Mamba_Hunt"
)

DATA_VIEW_ROOT = MAMBA_HUNT_ROOT / "RhythmMamba_DataView"
PREPROCESSED_ROOT = MAMBA_HUNT_ROOT / "RhythmMamba_Preprocessed"

PURE_DATA_PATH = DATA_VIEW_ROOT / "PURE"
UBFC_DATA_PATH = DATA_VIEW_ROOT / "UBFC"

PURE_CACHE_ROOT = PREPROCESSED_ROOT / "PURE"
UBFC_CACHE_ROOT = PREPROCESSED_ROOT / "UBFC"

LOCAL_INFERENCE_DIRECTORY = (
    PROJECT_ROOT / "configs" / "local" / "infer"
)

SCRIPTS_DIRECTORY = PROJECT_ROOT / "scripts"
RESULTS_ROOT = PROJECT_ROOT / "results"
LOG_DIRECTORY = RESULTS_ROOT / "logs"

for directory in [
    LOCAL_INFERENCE_DIRECTORY,
    SCRIPTS_DIRECTORY,
    RESULTS_ROOT / "models",
    RESULTS_ROOT / "runs",
    RESULTS_ROOT / "predictions",
    LOG_DIRECTORY,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# LOCAL INFERENCE CONFIG CREATOR
# ============================================================

def create_inference_config(
    source_config: Path,
    destination_config: Path,
    checkpoint_path: Path,
):
    with source_config.open("r", encoding="utf-8") as file:
        configuration = yaml.safe_load(file)

    for split_name in ["TRAIN", "VALID", "TEST"]:
        data_configuration = configuration[split_name]["DATA"]
        original_dataset_name = data_configuration["DATASET"]

        if original_dataset_name == "PURE":
            data_configuration["DATASET"] = "PURE"
            data_configuration["DATA_PATH"] = str(PURE_DATA_PATH)
            data_configuration["CACHED_PATH"] = str(PURE_CACHE_ROOT)

        elif original_dataset_name in ["UBFC", "UBFC-rPPG"]:
            # Official main.py recognizes the name "UBFC".
            data_configuration["DATASET"] = "UBFC"
            data_configuration["DATA_PATH"] = str(UBFC_DATA_PATH)
            data_configuration["CACHED_PATH"] = str(UBFC_CACHE_ROOT)

        else:
            raise ValueError(
                f"Unexpected dataset: {original_dataset_name}"
            )

        data_configuration["DO_PREPROCESS"] = False

    configuration["TOOLBOX_MODE"] = "only_test"
    configuration["DEVICE"] = "cuda:0"
    configuration["NUM_OF_GPU_TRAIN"] = 1

    configuration["INFERENCE"]["MODEL_PATH"] = str(
        checkpoint_path
    )

    configuration["LOG"]["PATH"] = str(
        RESULTS_ROOT / "runs"
    )

    configuration["MODEL"]["MODEL_DIR"] = str(
        RESULTS_ROOT / "models"
    )

    configuration["TEST"]["OUTPUT_SAVE_DIR"] = str(
        RESULTS_ROOT / "predictions"
    )

    with destination_config.open("w", encoding="utf-8") as file:
        yaml.safe_dump(
            configuration,
            file,
            sort_keys=False,
        )


# ============================================================
# PURE-TRAINED MODEL -> UBFC TEST
# ============================================================

pure_to_ubfc_config = (
    LOCAL_INFERENCE_DIRECTORY
    / "PURE_TO_UBFC_RHYTHMMAMBA_LOCAL.yaml"
)

create_inference_config(
    source_config=(
        OFFICIAL_ROOT
        / "configs"
        / "infer_configs"
        / "PURE_UBFC-rPPG_RHYTHMMAMBA.yaml"
    ),
    destination_config=pure_to_ubfc_config,
    checkpoint_path=(
        OFFICIAL_ROOT
        / "PreTrainedModels"
        / "PURE_cross_RhythmMamba.pth"
    ),
)


# ============================================================
# UBFC-TRAINED MODEL -> PURE TEST
# ============================================================

ubfc_to_pure_config = (
    LOCAL_INFERENCE_DIRECTORY
    / "UBFC_TO_PURE_RHYTHMMAMBA_LOCAL.yaml"
)

create_inference_config(
    source_config=(
        OFFICIAL_ROOT
        / "configs"
        / "infer_configs"
        / "UBFC-rPPG_PURE_RHYTHMMAMBA.yaml"
    ),
    destination_config=ubfc_to_pure_config,
    checkpoint_path=(
        OFFICIAL_ROOT
        / "PreTrainedModels"
        / "UBFC_cross_RhythmMamba.pth"
    ),
)


# ============================================================
# CREATE COMPATIBILITY LAUNCHER
# ============================================================

launcher_path = (
    SCRIPTS_DIRECTORY
    / "run_official_rhythmmamba.py"
)

launcher_code = '''#!/usr/bin/env python3

from pathlib import Path
import os
import runpy
import sys

import scipy.__config__ as scipy_config

PROJECT_ROOT = Path(__file__).resolve().parents[1]
OFFICIAL_ROOT = PROJECT_ROOT / "official" / "RhythmMamba"
OFFICIAL_MAIN = OFFICIAL_ROOT / "main.py"

if not OFFICIAL_MAIN.exists():
    raise FileNotFoundError(
        f"Official main.py not found: {OFFICIAL_MAIN}"
    )

# Compatibility for the unused MMPD loader import.
# This does not load or process the MMPD dataset.
if not hasattr(scipy_config, "get_info"):
    scipy_config.get_info = lambda *args, **kwargs: {}

if str(OFFICIAL_ROOT) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_ROOT))

# Required for the official relative resource paths.
os.chdir(OFFICIAL_ROOT)

# Forward all command-line arguments to official main.py.
sys.argv[0] = str(OFFICIAL_MAIN)

runpy.run_path(
    str(OFFICIAL_MAIN),
    run_name="__main__",
)
'''

launcher_path.write_text(
    launcher_code,
    encoding="utf-8",
)


# ============================================================
# VERIFY CREATED FILES
# ============================================================

created_configs = [
    pure_to_ubfc_config,
    ubfc_to_pure_config,
]

for config_path in created_configs:
    assert config_path.exists()

    with config_path.open("r", encoding="utf-8") as file:
        configuration = yaml.safe_load(file)

    checkpoint = Path(
        configuration["INFERENCE"]["MODEL_PATH"]
    )

    assert checkpoint.exists(), checkpoint
    assert configuration["TOOLBOX_MODE"] == "only_test"
    assert configuration["TEST"]["DATA"]["DO_PREPROCESS"] is False

    print("=" * 70)
    print("LOCAL INFERENCE CONFIGURATION")
    print("=" * 70)
    print("Configuration :", config_path)
    print("Train dataset :", configuration["TRAIN"]["DATA"]["DATASET"])
    print("Test dataset  :", configuration["TEST"]["DATA"]["DATASET"])
    print("Test cache    :", configuration["TEST"]["DATA"]["CACHED_PATH"])
    print("Checkpoint    :", checkpoint)
    print()

assert launcher_path.exists()

print("Compatibility launcher :", launcher_path)
print("\nLocal cross-dataset inference setup: PASSED")
print("Official RhythmMamba submodule remains unchanged.")

LOCAL INFERENCE CONFIGURATION
Configuration : /media/data/rPPG/Code/GitHub/Catch_The_Mamba/configs/local/infer/PURE_TO_UBFC_RHYTHMMAMBA_LOCAL.yaml
Train dataset : PURE
Test dataset  : UBFC
Test cache    : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/UBFC
Checkpoint    : /media/data/rPPG/Code/GitHub/Catch_The_Mamba/official/RhythmMamba/PreTrainedModels/PURE_cross_RhythmMamba.pth

LOCAL INFERENCE CONFIGURATION
Configuration : /media/data/rPPG/Code/GitHub/Catch_The_Mamba/configs/local/infer/UBFC_TO_PURE_RHYTHMMAMBA_LOCAL.yaml
Train dataset : UBFC
Test dataset  : PURE
Test cache    : /media/data/rPPG/rPPG_Data/Mamba_Hunt/RhythmMamba_Preprocessed/PURE
Checkpoint    : /media/data/rPPG/Code/GitHub/Catch_The_Mamba/official/RhythmMamba/PreTrainedModels/UBFC_cross_RhythmMamba.pth

Compatibility launcher : /media/data/rPPG/Code/GitHub/Catch_The_Mamba/scripts/run_official_rhythmmamba.py

Local cross-dataset inference setup: PASSED
Official RhythmMamba submodule remains unchanged.

# Run this in the terminal:

This performs inference only:
PURE→UBFC

| Metric  |   Paper | Your result |
| ------- | ------: | ----------: |
| MAE     |    0.95 |      0.9515 |
| RMSE    |    1.83 |      1.8257 |
| MAPE    |   1.04% |     1.0365% |
| Pearson |    0.99 |      0.9953 |
| SNR     | 6.35 dB |   6.3456 dB |


In [ ]:
conda activate mamba_hunting

cd /media/data/rPPG/Code/GitHub/Catch_The_Mamba

set -o pipefail

python scripts/run_official_rhythmmamba.py \
  --config_file /media/data/rPPG/Code/GitHub/Catch_The_Mamba/configs/local/infer/PURE_TO_UBFC_RHYTHMMAMBA_LOCAL.yaml \
  2>&1 | tee results/logs/PURE_TO_UBFC_pretrained.log


  100%|███████████████████████████████████████████| 42/42 [04:58<00:00,  7.11s/it]
FFT MAE (FFT Label): 0.9515237238009514 +/- 0.24042822143851042
FFT RMSE (FFT Label): 1.8257157617897743 +/- 1.741639660820831
FFT MAPE (FFT Label): 1.0364803085887064 +/- 0.2864522388156272
FFT Pearson (FFT Label): 0.9953426152214278 +/- 0.015242275358496196
FFT SNR (FFT Label): 6.345639714581549 +/- 1.193754541758722 (dB)
(mamba_hunting) rafsan@user-GPU:/media/data/rPPG/Code/GitHub/Catch_The_Mamba$ 


UBFC-trained checkpoint → PURE

| Metric  | Expected |
| ------- | -------: |
| MAE     | 1.98 BPM |
| RMSE    | 6.51 BPM |
| MAPE    |    3.59% |
| Pearson |     0.96 |
| SNR     |  8.94 dB |
